## **Local Setup**

* Follow the **[instructions](../README.md)** to set up your local environment.
* If you are launching the notebook on VS Code, select the kernel starting with `pfdl`.
* To directly launch the notebook on a localhost via terminal, run `uv jupyter-lab notebooks/01_simple_mlp_model.ipynb` inside the project root.
* Skip **[Google Colab Setup](#google-colab-setup)** below.

In [ ]:
%load_ext autoreload
%autoreload 2

---

## **Google Colab Setup**

* Skip **[Local Setup](#local-setup)** section.
* Uncomment the cell below and execute the code block to install all dependencies and resolve import paths.
* After successful installation, restart your runtime session to clear Colab's previous Python cache.

>**Note**: To install CUDA-enabled PyTorch, first switch to GPU session and replace `uv pip install -e .[cpu,dev] ...` with `uv pip install -e .[gpu,dev] ...` in the following cell and rerun the

In [ ]:
# # Clone the repo locally
# !rm -rf /content/pfdl
# !git clone https://github.com/paymantohidifar/protein-function-deep-learning.git --branch dev pfdl
# %cd pfdl

# # Bootstrap uv globally and pull GPU-enabled binaries directly into the system layer
# !curl -LsSf https://astral.sh/uv/install.sh | sh && \
# export PATH="$HOME/.local/bin:${PATH}" && \
# uv pip install -e .[cpu,dev] \
#         --system \
#         --break-system-packages \
#         --color never

# # Add `src/` to system path for local imports
# import sys
# sys.path.append('/content/pfdl/src') # Point to the cloned directory

## **Project Directories Setup**

In [ ]:
from pathlib import Path
import sys

if 'google.colab' in sys.modules:
    root_dir = Path("/content")
else:
    root_dir = Path.cwd().resolve().parent

dataset_dir = root_dir / "dataset"
models_dir = root_dir / "models"

print(root_dir)
print(dataset_dir)
print(models_dir)

---

# **ESM-2 Powered Deep Leaning Model for Protein Function Prediction**

---

## **Table of Contents**

---

## **Introduction**

Our goal in this notebook is to build a simple neural network model to predict protein function from its amino acid sequence. We will use the [CAFA (Critical Assessment of Functional Annotation)](https://biofunctionprediction.org/cafa/) dataset as our primary resource and will conduct all necessary steps from data preparation, model training to model performance evaluation. In the end, we will briefly discuss potential options to improve model performance further.

---

## **Preparing the Data**

In this section, we will follow steps in the following workflow to turn raw biological data into a model-ready format:

1. **Sequence Standardization:** Cleaning and validating amino acid strings.
2. **Functional Mapping:** Associating sequences with hierarchical GO (Gene Ontology) terms.
3. **Vectorization:** Converting text-based sequences into numerical embeddings using models like ESM-2.
4. **Filtering:** Removing overly broad GO terms to ensure the model learns specific biological insights.

Note that the step-by-step approach above will ensure that when our model struggles, we have the intuition to identify whether the flaw lies in the learning process or the biological data itself.

### **Loading CAFA3 Data**

There have been several rounds of [CAFA](https://biofunctionprediction.org/cafa/), but the CAFA3 dataset (2016-2017) is the most recent publicly available one. You can find more about CAFA3 [here](https://link.springer.com/article/10.1186/s13059-019-1835-8).

We will first download the “CAFA3 Targets” and “CAFA3 Training Data” files from the CAFA website.

Links to CAFA4 datasets (2019-2020) are currently broken on the source site. However, if you are interested in more recent datasets, [CAFA5](https://www.kaggle.com/c/cafa-5-protein-function-prediction) and [CAFA6](https://www.kaggle.com/competitions/cafa-6-protein-function-prediction/data?select=Train) are avialable through Kaggle competitions.

#### **Original CAFA3 Dataset**

In [ ]:
# cafa3_train_data_url = "https://biofunctionprediction.org/cafa-targets/CAFA3_training_data.tgz"

#### **CAFA3 Dataset Prepared for DBFB**

In [ ]:
from pfdl.downloads import download_data

dataset_subdir = dataset_dir / "dlfb-version"
filename = "train_terms.tsv.zip"
destination = dataset_subdir / filename

base_url = "https://assets.deep-learning-for-biology.com/proteins/datasets"

dataset_subdir.mkdir(parents=True, exist_ok=True)
if destination.exists():
    print("File already exists! Skipping the download.")
else:
    download_data(base_url, filename, destination)

In [ ]:
import pandas as pd

labels = pd.read_csv(
    destination, sep='\t', compression='infer'
)

print(labels.shape)
labels.head()

The dataframe above has three columns:
* **EntryID:** UniProt ID of the protein
* **term:** A GO accession code describing a specific protein function
* **aspect:** The GO category the function belongs to (e.g. biological process (BPO), molecular function (MFO), and cellular component (CCO))

### **Adding Human-readable GO Terms**

To make CAFA dataset more human-readable, we cross-reference GO terms using the official Gene Ontology library. For more details, check [this](https://geneontology.org/docs/download-go-annotations/) out.

We will use `obonet` Python library to parse `.obo` (Open Biomedical Ontology format) graph format to programmatically map each GO ID to its biological definition using `get_go_term_descriptions` helper fucntion:

In [ ]:
from pfdl.dataset import get_go_term_descriptions

dataset_subdir = dataset_dir / "dlfb-version"
filename = "go_term_descriptions.zip"
destination = dataset_subdir / filename

obo_url = "https://current.geneontology.org/ontology/go-basic.obo"

go_term_descriptions = get_go_term_descriptions(obo_url, store_path=destination)

In [ ]:
print(go_term_descriptions.shape)
go_term_descriptions.head()

We will add `description` columns to `labels` by merging the two dataframes on `term` column:

In [ ]:
labels = labels.merge(go_term_descriptions, on='term')

print(labels.shape)
labels.head()

We can see that we lost some of rows in the original labels dataframe because some of Entry IDs were missing in `go_term_descriptions` dataframe. We should be fine even losing some of the data.

In this work, we will focus on **Molecular Function** terms denoted as **MFO** on `aspect` column. Here is the example counts on each GO category:

In [ ]:
print(labels.value_counts('aspect'))

In [ ]:
labels = labels[labels['aspect'] == "MFO"]

print(labels.shape)

To gain some insight about different types of Molecular Function GO terms, let's quickly examine their distributions:

In [ ]:
labels.value_counts('description')

As shown above, the distribution of protein function annotations is heavily skewed, dominated by generic terms like "molecular_function" and "binding." These high-frequency labels provide minimal biological insight and will be filtered out later to improve model performance.

#### **Adding Protein Sequeces**
To pair these labels with their respective biological data, we will dowload and load protein sequences from a FASTA file. Using the `BioPython.SeqIO` module, we can parse these sequences into a Pandas DataFrame for easier manipulation.

In [ ]:
dataset_subdir = dataset_dir / "dlfb-version"
filename = "train_sequences.fasta"
destination = dataset_subdir / filename

base_url = "https://assets.deep-learning-for-biology.com/proteins/datasets"

dataset_subdir.mkdir(exist_ok=True)
if destination.exists():
    print("File already exists! Skipping the download.")
else:
    download_data(base_url, filename, destination)

In [ ]:
from Bio import SeqIO

records = SeqIO.parse(open(destination), format='fasta')
sequence_df = pd.DataFrame(
    data=[(record.id, str(record.seq), len(record.seq)) for record in records],
    columns=["EntryID", "Sequence", "Length"]
)

print(sequence_df.shape)
sequence_df.head()

The CAFA dataset includes proteins from many different organisms. In this notebook, we will focus on human proteins. To retrieve human proteins, we will use NCBI taxonomy ("Tax" for short) IDs associated with them. This information is available in `train_taxonomy.tsv.zip` file.

We need to download the file and add the Tax IDs to `sequence_df` and keep examples with Homo sapiens Tax ID **9606**.

In [ ]:
dataset_subdir = dataset_dir / "dlfb-version"
filename = "train_taxonomy.tsv.zip"
destination = dataset_subdir / filename

base_url = "https://assets.deep-learning-for-biology.com/proteins/datasets"

dataset_subdir.mkdir(exist_ok=True)
if destination.exists():
    print("File already exists! Skipping the download.")
else:
    download_data(base_url, filename, destination)

In [ ]:
taxonomy = pd.read_csv(destination, sep='\t', compression='infer')

print(taxonomy.shape)
taxonomy.head()

In [ ]:
sequence_df = sequence_df.merge(taxonomy, on='EntryID')

print(sequence_df.shape)
sequence_df.head()

In [ ]:
sequence_df = sequence_df[sequence_df['taxonomyID'] == 9606]

print(sequence_df.shape)
sequence_df.head()

We ended up with about 25K human proteins. Now, add Go terms on `labels` dataframe to our human `sequence_df` dataframe"

In [ ]:
sequence_df = sequence_df.merge(labels, on='EntryID')

print(sequence_df.shape)

print(
    f"Dataset contains {sequence_df['EntryID'].nunique()} proteins with "
    f"{sequence_df['term'].nunique()} molecular functions."
)

We can already see that many proteins are associated with multiple molecular functions. To quantify this, we examine the distribution of the number of functions per protein:

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
sequence_df.groupby('EntryID')['term'].nunique().hist(bins=50, log=True, ax=ax)
ax.set_xlabel("Number of molecular function annotation per protein")
ax.set_ylabel("Frequency");

The biological reality of protein "multitasking" means that many proteins perform multiple roles, acting as enzymes while simultaneously binding to other molecules. For machine learning, this creates two specific challenges: multi-label classification and extreme class imbalance (where some functions are rare and others are ubiquitous).

As we showed in an ealier step, broad terms like "molecular function" or "protein binding" are so universal that they provide almost no predictive value. To prevent the model from "cheating" by fixating on these dominant but generic labels, we must explicitly filter them out during preprocessing, forcing the model to learn more specific and meaningful biological functions.

In [ ]:
print(sequence_df['description'].value_counts())

In [ ]:
# Retrieve top 3 generic functions
generic_functions = sequence_df['description'].value_counts().head(3).index.tolist()

# Remove examples associated with `generic functions`
sequence_df = sequence_df[~sequence_df['description'].isin(generic_functions)]

print(sequence_df.shape)

On the other hand, we see many molecular functions are exceptionally rare; for instance, BRE binding appears only a single time in the dataset.

To ensure the model learns statistically significant associations rather than noise, we must provide sufficient training examples for each functional category. We will keep the examples with molecular function associated with at least **50** distrinct proteins. Note that this number can be assumed to be a hyperparameter.

In [ ]:
# Retrieve "GO terms" with at least 50 distict proteins
common_functions = sequence_df['term'].value_counts()[sequence_df['term'].value_counts() > 50].index

# Keep examples associated with `common_functions`
sequence_df = sequence_df[sequence_df['term'].isin(common_functions)]

print(sequence_df.shape)

# Verify
print(sequence_df['term'].value_counts())

We need to reshape our dataset to be able to separate features (e.g. sequence embeddings, length, etc) from targets (protein function or GO term) by transforming the categorical protein function annotations into a binary occurrence matrix, analogous to one-hot encoding.

We will ignore certain columns like `taxonomyID`, `aspect`, and `description` as they don't provide additional information.

In [ ]:
# Keep important columns, add `value` column with 1's,
# pivot the table on `term` column
sequence_df_wide = (
    sequence_df[['EntryID', 'Sequence', 'Length', 'term']]
    .assign(value=1)
    .pivot(
        index=['EntryID', 'Sequence', 'Length'],
        columns='term',
        values='value'
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

print(sequence_df_wide.shape)
sequence_df_wide.head()

This dataset is now in a format that’s almost ready for machine learning.

Before we move on, let’s run a few final sanity checks on (1) number of unique proteins (2) number of unique sequences:

In [ ]:
print("Number of unique proteins:", sequence_df_wide['EntryID'].nunique())
print("Number of unique sequences:", sequence_df_wide['Sequence'].nunique())

The number of protein in 10,000 is in right ballpark. There are roughly 21,000 protein-coding genes in the human genome, and since we applied several filtering steps, we expect a somewhat smaller number. It’s always worth keeping rough order-of-magnitude expectations in mind—if we saw 1,000 or 1,000,000 here, we’d suspect something was off.

On the other hand, number of sequences is a little less than the number of proteins, suggesting some of the proteins have identical sequences.

#### **Capping Protein Sequences with Upper Length Limit**

Downstream machine/deep learning models cannot directly ingest variable-length raw text representations of protein sequences. To bridge this gap, we need to project these sequences into continuous, dense numerical vector spaces using protein language models, such as ESM-2 (See [here](#converting-protein-sequences-to-their-mean-enbeddings)).

The self-attention engine in ESM-2 Transformer models scales quadratically in both memory consumption and computational complexity relative to the sequence length, $O(L^2)$, and long sequences can easily trigger Out-Of-Memory (OOM) errors during inference. Therefore, we need to apply an upper limit to our sequence array lengths.

But, what is a good length not to surpass our hardware limits and retain sufficient amount of data for training our model? We will find this out by visualizing the absolute sequence length distribution across our examples:

In [ ]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(5, 4))
sns.histplot(ax=ax, data=sequence_df[sequence_df['Length'] < 5000], x='Length', bins=100);

print("Median of length: ", sequence_df_wide['Length'].quantile(0.5))
print("75% quantile of length: ", sequence_df_wide['Length'].quantile(0.75))

Based on the figure and quantiles, maximum of 500 amino acids can be a good heuristic to start with:

In [ ]:
sequence_df_wide = sequence_df_wide[sequence_df_wide['Length'] <= 500]
print(sequence_df_wide.shape)

We almost halved the dataset size which is perfectly fine for our initial prototyping.

### **Splitting the Dataset into Train/Validation/Test Sets**

To ensure reliable evaluation, we will partition our examples by `EntryID` into three mutually exclusive train (60%), validation (20%), and test (20%) subsets:

In [ ]:
from sklearn.model_selection import train_test_split

train_seq_ids, valid_test_seq_id = train_test_split(
    list(set(sequence_df_wide['EntryID'])), test_size=0.4, random_state=42
)

valid_seq_ids, test_seq_ids = train_test_split(
    valid_test_seq_id, test_size=0.5, random_state=42
)

sequence_splits = {
    'train': sequence_df_wide[sequence_df_wide['EntryID'].isin(train_seq_ids)],
    'valid': sequence_df_wide[sequence_df_wide['EntryID'].isin(valid_seq_ids)],
    'test': sequence_df_wide[sequence_df_wide['EntryID'].isin(test_seq_ids)],
}

for subset, data in sequence_splits.items():
    print(f"{subset} has {len(data)} entries.")

### **Converting Protein Sequences to Their Mean Enbeddings**

As we touched on sequence embeddings in the previous subsectiopn, we will use ESM-2 to generate rich representation of our protein sequences. Briefly, EMS-2 is trained via masked language modeling across millions of evolutionarily diverse sequences and leverages deeply stacked self-attention layers to generate contextualized representations that implicitly capture spatial structural topography and thermodynamic properties without requiring explicit 3D coordinates.

We will now convert the protein sequences from each split into their mean embeddings. To optimize this time-intensive process, we utilize two strategies: (1) optionally using a GPU to parallelize the transformer forward pass, (2) computing the embeddings once and saving them to disk (serialization) to avoid redundant computation in future sessions.

To streamline this, we will use two helper functions `store_sequence_embeddings` for storing and `load_sequence_embeddings` loading these high-dimensional vectors. `store_sequence_embeddings` utilize another helper function to return mean embedding for each sequence using a protein LM.

In [ ]:
from transformers import AutoTokenizer, EsmModel

model_checkpoint = "facebook/esm2_t30_150M_UR50D" # "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, cache_dir=models_dir)
model = EsmModel.from_pretrained(model_checkpoint, cache_dir=models_dir)

In [ ]:
from pfdl.dataset import store_sequence_embeddings

for split, df in sequence_splits.items():
    store_sequence_embeddings(
        sequence_df=df,
        store_prefix=str(dataset_subdir / f"protein_dataset_{split}"),
        tokenizer=tokenizer,
        model=model,
    )

In [ ]:
from pfdl.dataset import load_sequence_embeddings

train_df = load_sequence_embeddings(
    store_file_prefix=str(dataset_subdir / "protein_dataset_train"),
    model_checkpoint=model_checkpoint
)

print(train_df.shape)
train_df.head()

On `train_df`, we see a series of columns labeled `ME:1` through `ME:640`. These represent the **mean-pooled hidden states** from the last hidden layer of the ESM2 model, effectively a fixed-length numerical summary of each protein sequence. These embeddings capture biochemical and structural information learned during pretraining and will serve as the input features for our classifier.

We will use a custom PyTorch Dataset class `ProtDataset` as well as a helper function `create_data_loader` to prepare the datasets for each split. Here is an example for `train_df`. We apply it to all three subsets using `build_dataset` helper function.

In [ ]:
from pfdl.dataset import ProtDataset, create_data_loader

ds = ProtDataset(train_df)
data_loader = create_data_loader(ds, batch_size=32, is_training=True)
batch = next(iter(data_loader))
print(batch['embedding'].shape, batch['target'].shape)

In [ ]:
from pfdl.dataset import build_dataset

dataset_splits = build_dataset(
    store_file_prefix=str(dataset_subdir / "protein_dataset"),
    model_checkpoint=model_checkpoint
)

With this, we now have our data fully preprocessed and ready to use in training a model.

---

## **Training the Model**

In this notebook, we will train a lightweight Multi-Layer Perceptron (MLP) on top of the fixed-size protein embeddings. While the original protein sequences vary in length, our pre-computed mean embeddings provide a consistent input for the model.

Our objective is to predict associations across 291 molecular functions. This is a multi-label classification task, as a single protein often performs multiple biological roles simultaneously.

Again, in this notebook, the ESM2 model remains frozen; we are not fine-tuning its internal parameters. Instead, our MLP acts as a task-specific head that learns to interpret the static representations generated by ESM2.

---

### **Evaluating the Model**

----

## **Summary**

---

## **Next Steps**